In [ ]:
# print("123")

123


In [1]:
!pip install opentelemetry-api

In [2]:
!pip install opentelemetry-sdk

In [6]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)


In [3]:
!pip install gitsource

In [4]:
!pip install minsearch

In [5]:
!pip install google-genai

In [8]:
import os
import sqlite3
import time

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)
from starter import rag

In [9]:
# instrumentation created at import time is backed by our provider.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [10]:
INPUT_PRICE_PER_MILLION = 0.15
OUTPUT_PRICE_PER_MILLION = 0.60

In [11]:
class RAGTraced:
    """Wraps a RAGBase instance so rag(), search(), and llm() each
    produce their own OTel span."""

    def __init__(self, rag_base, tracer):
        self._rag = rag_base
        self._tracer = tracer

    def search(self, query, num_results=5):
        with self._tracer.start_as_current_span("search") as span:
            results = self._rag.search(query, num_results=num_results)
            span.set_attribute("num_results", len(results))
            return results

    def llm(self, prompt):
        with self._tracer.start_as_current_span("llm") as span:
            response = self._rag.llm(prompt)
            usage = response.usage

            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            input_cost = (usage.input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
            output_cost = (usage.output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
            cost = input_cost + output_cost
            span.set_attribute("cost", cost)

            return response

    def build_prompt(self, query, search_results):
        return self._rag.build_prompt(query, search_results)

    def rag(self, query):
        with self._tracer.start_as_current_span("rag") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            return response.output_text


In [12]:
QUERY = "How does the agentic loop keep calling the model until it stops?"

In [13]:
traced_rag = RAGTraced(rag, tracer)

In [ ]:
print("=" * 70)
print("Q1-Q3: Running traced RAG with ConsoleSpanExporter")
print("=" * 70)

answer = traced_rag.rag(QUERY)
provider.force_flush()

print("\nAnswer:", answer[:200], "...")
print(
    "\n(Inspect the ReadableSpan dicts printed above to answer Q1 [span count], "
    "Q2 [llm span's input_tokens attribute], and Q3 [search vs llm span duration])"
)

Q1-Q3: Running traced RAG with ConsoleSpanExporter
{
    "name": "search",
    "context": {
        "trace_id": "0x4c7c15dd45393c37f1607b8af10edd51",
        "span_id": "0xa029680261b1beec",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x27638caccf350899",
    "start_time": "2026-07-26T15:39:15.273729Z",
    "end_time": "2026-07-26T15:39:15.306089Z",
    "status": {
        "status_code": "ERROR",
        "description": "FilterValidationError: Unknown filter field 'course'. Valid fields are: ['filename']"
    },
    "attributes": {},
    "events": [
        {
            "name": "exception",
            "timestamp": "2026-07-26T15:39:15.306052Z",
            "attributes": {
                "exception.type": "minsearch.filters.validator.FilterValidationError",
                "exception.message": "Unknown filter field 'course'. Valid fields are: ['filename']",
                "exception.stacktrace": "Traceback (most recent call last):\n  File \"c

FilterValidationError: Unknown filter field 'course'. Valid fields are: ['filename']